In [2]:
# ============================================================
# REAL ESTATE PRICE INTELLIGENCE SYSTEM
# Author: Sumeet Rajput.
# ============================================================

# =========================
# IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import statsmodels.api as sm
from scipy.stats import skew, f_oneway
from statsmodels.stats.outliers_influence import variance_inflation_factor

%matplotlib inline
plt.ioff()
sns.set_style("whitegrid")

# ============================================================
# 1️⃣ LOAD DATA
# ============================================================
df = pd.read_csv("housing_data.csv")
df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

# ---------- DATA TYPE CORRECTIONS ----------

# Convert month names to numeric month
month_map = {
    "Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
    "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
}
df["MoSold"] = df["MoSold"].map(month_map)

# MSSubClass is actually numeric category
df["MSSubClass"] = df["MSSubClass"].str.replace("SC", "", regex=False)
df["MSSubClass"] = pd.to_numeric(df["MSSubClass"], errors="coerce")

# Create a true datetime column (VERY useful later)
df["SaleDate"] = pd.to_datetime(
    df["YrSold"].astype(str) + "-" + df["MoSold"].astype(str) + "-01"
)

display(df.head())
display(df.info())


# ============================================================
# 2️⃣ DATA CLEANING
# ============================================================

# ---------- NUMERIC MISSING ----------
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns
df[numeric_cols] = df[numeric_cols].apply(lambda x: x.fillna(x.median()))

# ---------- CATEGORICAL MISSING ----------
cat_cols = df.select_dtypes(include="object").columns
df[cat_cols] = df[cat_cols].fillna("None")

# Special case: houses without garage
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove extreme outliers (Ames dataset known issue)
df = df[df["GrLivArea"] < 4000]

print("Cleaning Completed. Rows left:", len(df))


# ============================================================
# 3️⃣ FEATURE ENGINEERING
# ============================================================
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["PricePerSqFt"] = df["SalePrice"] / df["GrLivArea"]
df["PricePerSqFt"] = df["PricePerSqFt"].replace([np.inf, -np.inf], np.nan)
df["PricePerSqFt"] = df["PricePerSqFt"].fillna(df["PricePerSqFt"].median())
df["LogSalePrice"] = np.log1p(df["SalePrice"])
df["Size_Quality"] = df["GrLivArea"] * df["OverallQual"]

# Price Segmentation
df["PriceSegment"] = pd.qcut(
    df["SalePrice"],
    4,
    labels=["Budget","Mid","Premium","Luxury"]
)

# ============================================================
# HELPER FUNCTION
# ============================================================
def new_fig(title, insight):
    clear_output(wait=True)
    plt.close('all')
    display(Markdown(f"## {title}"))
    display(Markdown(f"**Insight:** {insight}"))
    fig, ax = plt.subplots(figsize=(12,6))
    return fig, ax

# ============================================================
# 4️⃣ UNIVARIATE ANALYSIS
# ============================================================
def univariate_analysis():
    fig, ax = new_fig(
        "Market Price Distribution",
        f"Skewness = {round(skew(df['SalePrice']),2)} → Premium outliers present."
    )
    sns.histplot(df["SalePrice"], kde=True, ax=ax)
    plt.show()

# ============================================================
# 5️⃣ SIZE IMPACT ANALYSIS
# ============================================================
def size_analysis():
    fig, ax = new_fig(
        "Size vs Price",
        "Larger homes command higher prices, but price growth slows for very large houses."
    )

    sns.regplot(
        data=df,
        x="GrLivArea",
        y="SalePrice",
        scatter_kws={"alpha":0.4},
        line_kws={"color":"red"},
        ax=ax
    )

    ax.set_xlabel("Above Ground Living Area (Sq Ft)")
    ax.set_ylabel("Sale Price")
    plt.show()


# ============================================================
# 6️⃣ BEDROOM & BATHROOM IMPACT
# ============================================================
def bedroom_bath_analysis():
    fig, ax = new_fig(
        "Bedrooms Impact on Pricing",
        "More bedrooms increase price up to optimal range."
    )
    sns.boxplot(x="BedroomAbvGr", y="SalePrice", data=df, ax=ax)
    plt.show()

# ============================================================
# 7️⃣ AMENITIES IMPACT
# ============================================================
def amenities_analysis():
    fig, ax = new_fig(
        "Garage Capacity Impact",
        "Garage capacity significantly influences pricing."
    )
    sns.boxplot(x="GarageCars", y="SalePrice", data=df, ax=ax)
    plt.show()

# ============================================================
# 8️⃣ LOCATION PREMIUM ANALYSIS
# ============================================================
def location_analysis():
    fig, ax = new_fig(
        "Neighborhood Price Premium",
        "Clear geographic pricing disparities detected."
    )
    location_price = df.groupby("Neighborhood")["SalePrice"].median().sort_values()
    location_price.plot(kind="barh", ax=ax)
    plt.show()

# ============================================================
# 9️⃣ MARKET TREND ANALYSIS
# ============================================================
def market_trend():
    fig, ax = new_fig(
        "Historical Market Trend",
        "Market appreciation trend observed over time."
    )
    trend = df.groupby("YrSold")["SalePrice"].mean()
    trend.plot(marker="o", ax=ax)
    plt.show()

# ============================================================
# 🔟 CORRELATION ANALYSIS
# ============================================================
def correlation_analysis():
    fig, ax = new_fig(
        "Feature Correlation Matrix",
        "Quality and Size are strongest predictors."
    )
    corr = df[["SalePrice","GrLivArea","OverallQual",
               "GarageCars","HouseAge","PricePerSqFt"]].corr()
    sns.heatmap(corr, annot=True, cmap="coolwarm", ax=ax)
    plt.show()

# ============================================================
# 1️⃣1️⃣ REGRESSION MODEL
# ============================================================
def regression_model():
    features = ["GrLivArea","OverallQual","GarageCars",
                "HouseAge","Size_Quality"]
    X = sm.add_constant(df[features])
    y = df["LogSalePrice"]

    model = sm.OLS(y,X).fit()
    display(model.summary())

# ============================================================
# 1️⃣2️⃣ VIF CHECK
# ============================================================
def vif_check():
    features = ["GrLivArea","OverallQual","GarageCars","HouseAge"]
    X = df[features]
    vif = pd.DataFrame()
    vif["Feature"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values,i)
                  for i in range(len(X.columns))]
    display(vif)

# ============================================================
# 1️⃣3️⃣ INVESTMENT OPPORTUNITY ANALYSIS
# ============================================================
def investment_analysis():
    features = ["GrLivArea","OverallQual","GarageCars","HouseAge"]
    X = sm.add_constant(df[features])
    model = sm.OLS(df["LogSalePrice"],X).fit()

    df["PredictedPrice"] = np.expm1(model.predict(X))
    df["PriceGap"] = df["SalePrice"] - df["PredictedPrice"]

    fig, ax = new_fig(
        "Undervalued vs Overvalued Properties",
        "Negative gap → Investment opportunity."
    )
    sns.histplot(df["PriceGap"], kde=True, ax=ax)
    plt.show()

# ============================================================
# 1️⃣4️⃣ MARKET SEGMENTATION (CLUSTERING)
# ============================================================
def segmentation():
    features = df[["GrLivArea","OverallQual",
                   "GarageCars","PricePerSqFt"]]
    scaled = StandardScaler().fit_transform(features)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df["Segment"] = kmeans.fit_predict(scaled)

    fig, ax = new_fig(
        "Buyer Persona Segmentation",
        "Clusters represent Budget, Family, Premium, Luxury buyers."
    )
    sns.scatterplot(x="GrLivArea",
                    y="SalePrice",
                    hue="Segment",
                    palette="Set2",
                    data=df,
                    ax=ax)
    plt.show()

    display(df.groupby("Segment")[["SalePrice","GrLivArea",
                                   "OverallQual"]].mean())

# ============================================================
# DASHBOARD INTERFACE
# ============================================================
analysis_dict = {
    "Univariate Analysis": univariate_analysis,
    "Size Impact": size_analysis,
    "Bedroom Impact": bedroom_bath_analysis,
    "Amenities Impact": amenities_analysis,
    "Location Premium": location_analysis,
    "Market Trend": market_trend,
    "Correlation Analysis": correlation_analysis,
    "Regression Model": regression_model,
    "VIF Stability": vif_check,
    "Investment Opportunities": investment_analysis,
    "Market Segmentation": segmentation
}

dropdown = widgets.Dropdown(
    options=list(analysis_dict.keys()),
    description="Select Analysis:",
    layout=widgets.Layout(width="60%")
)

output = widgets.Output()

def update(change):
    with output:
        analysis_dict[change["new"]]()

dropdown.observe(update, names="value")

display(dropdown, output)
update({"new": list(analysis_dict.keys())[0]})


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,SaleDate
0,60,RL,65,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,No,No,No,0,2,2008,WD,Normal,208500,2008-02-01
1,20,RL,80,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,No,No,No,0,5,2007,WD,Normal,181500,2007-05-01
2,60,RL,68,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,No,No,No,0,9,2008,WD,Normal,223500,2008-09-01
3,70,RL,60,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,No,No,No,0,2,2006,WD,Abnorml,140000,2006-02-01
4,60,RL,84,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,No,No,No,0,12,2008,WD,Normal,250000,2008-12-01


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   MSSubClass     1460 non-null   int64         
 1   MSZoning       1460 non-null   object        
 2   LotFrontage    1460 non-null   int64         
 3   LotArea        1460 non-null   int64         
 4   Street         1460 non-null   object        
 5   Alley          91 non-null     object        
 6   LotShape       1460 non-null   object        
 7   LandContour    1460 non-null   object        
 8   Utilities      1460 non-null   object        
 9   LotConfig      1460 non-null   object        
 10  LandSlope      1460 non-null   object        
 11  Neighborhood   1460 non-null   object        
 12  Condition1     1460 non-null   object        
 13  Condition2     1460 non-null   object        
 14  BldgType       1460 non-null   object        
 15  HouseStyle     1460 n

None

Cleaning Completed. Rows left: 1456


Dropdown(description='Select Analysis:', layout=Layout(width='60%'), options=('Univariate Analysis', 'Size Imp…

Output()